In [ ]:
!pip install folium

In [ ]:
import folium
import pandas as pd
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"

spacex_df = pd.read_csv(URL)

spacex_df.head()

In [ ]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

### Task 1: Mark all launch sites on a map

In [ ]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

In [ ]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [ ]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [ ]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label


In [ ]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add a circle and marker for each launch site
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    label = row['Launch Site']

    # Circle
    folium.Circle(
        coordinate,
        radius=1000,
        color='#d35400',
        fill=True
    ).add_child(
        folium.Popup(label)
    ).add_to(site_map)

    # Marker
    folium.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % label
        )
    ).add_to(site_map)

site_map

### Question and Answer
1. Are all launch sites in proximity to the Equator line?

No.

The launch sites are located approximately between 28°N and 35°N latitude:

CCAFS SLC-40 → ~28.56°N
KSC LC-39A → ~28.61°N
VAFB SLC-4E → ~34.63°N

The Equator is at 0° latitude, so these sites are thousands of kilometers north of it. However, they are still located in relatively low latitudes compared to many other places in the United States.

Reason: Launching closer to the Equator provides an extra boost from Earth's rotational speed, reducing fuel requirements. This is why many launch facilities are built in southern regions.

2. Are all launch sites in very close proximity to the coast?

Yes.

From the map:

CCAFS SLC-40 and KSC LC-39A are on Florida's east coast.
VAFB SLC-4E is on the California coast.

All launch sites are located very near the ocean.

Reason: Coastal locations are safer for launches because rockets travel over open water. If debris falls during launch or stage separation, it is less likely to threaten populated areas. Coastal sites also provide wider launch corridors for different orbital inclinations.

### Findings
SpaceX launch sites are not near the Equator, but they are at relatively low latitudes.
All launch sites are very close to the coast.
Coastal locations improve safety, while lower latitudes improve launch efficiency by taking advantage of Earth's rotation.

### Task 2 

In [ ]:
spacex_df.tail(10)

In [ ]:
marker_cluster = MarkerCluster()

In [ ]:
# Create marker colors based on launch success/failure
spacex_df['marker_color'] = spacex_df['class'].map({
    1: 'green',
    0: 'red'
})

spacex_df[['class', 'marker_color']].head()

In [ ]:
from folium.plugins import MarkerCluster

marker_cluster = MarkerCluster()

In [ ]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    # marker = folium.Marker(...)
    marker_cluster.add_child(marker)

site_map

In [ ]:
# Add marker cluster to the map
site_map.add_child(marker_cluster)

# Add a marker for each launch
for index, record in spacex_df.iterrows():

    coordinate = [record['Lat'], record['Long']]

    marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(
            color='white',
            icon_color=record['marker_color']
        ),
        popup=record['Launch Site']
    )

    marker_cluster.add_child(marker)

site_map

### Task 3 
Calculate the distances between a launch site to its proximities

In [ ]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [ ]:
launch_site_lat = 28.563197
launch_site_lon = -80.576820

In [ ]:
coastline_lat = 28.56367
coastline_lon = -80.57163

In [ ]:
distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon
)

distance_coastline

In [ ]:
coastline_coordinate = [coastline_lat, coastline_lon]

distance_marker = folium.Marker(
    coastline_coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:.2f} KM</b></div>'.format(distance_coastline)
    )
)

site_map.add_child(distance_marker)

In [ ]:
lines = folium.PolyLine(
    locations=[
        [launch_site_lat, launch_site_lon],
        [coastline_lat, coastline_lon]
    ],
    weight=2
)

site_map.add_child(lines)

site_map

In [ ]:
highway_lat = 28.56350
highway_lon = -80.57000

distance_highway = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    highway_lat,
    highway_lon
)

folium.Marker(
    [highway_lat, highway_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size:12px;"><b>{:.2f} KM</b></div>'.format(distance_highway)
    )
).add_to(site_map)

folium.PolyLine(
    [[launch_site_lat, launch_site_lon],
     [highway_lat, highway_lon]]
).add_to(site_map)

### Answer and Finding
- Are launch sites in close proximity to railways?

No. Most launch sites are not located very close to railways. Railways are not essential for launch operations and could introduce safety concerns near launch facilities.

- Are launch sites in close proximity to highways?

Yes. Launch sites are generally near major highways. Highways facilitate transportation of personnel, equipment, and rocket components.

- Are launch sites in close proximity to coastline?

Yes. All SpaceX launch sites are located near coastlines. This allows rockets to launch over open water, reducing risk to populated areas in case of debris or launch failure.

- Do launch sites keep certain distance away from cities?

Yes. Launch facilities are intentionally built away from densely populated cities. Rocket launches involve noise, vibrations, hazardous materials, and potential accidents, making remote locations safer.

### Findings
- Launch sites are very close to coastlines.
- Launch sites are reasonably accessible via highways.
- Launch sites are far from major cities for safety reasons.
- Railways are generally not a significant factor in launch site selection.
- The locations balance safety, logistics, and launch efficiency.